[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_06_deep_learning/ch32_attention_transformers_modern_ai/03_coding_exam_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 32 Part VI Capstone Coding Exam — Transformers, Clinical NLP & LoRA (100 Points)

**Book Reference:** Chapter 32, Section 32.7 (Part VI Capstone Coding Exam)  
**Suggested Time:** 50–60 minutes (Open Book, Closed Solution)

| Task | Transformer & Foundation-Model Deliverable | Points |
| :--- | :--- | :---: |
| **Task 32.1** | `verify_causal_attention_mask(seq_len)` (Autoregressive lower-triangular mask audit) | 30 |
| **Task 32.2** | `train_clinical_note_transformer(seed=42, epochs=25)` (End-to-end clinical triage Transformer achieving `val_auc >= 0.90`) | 35 |
| **Task 32.3** | `audit_lora_parameter_efficiency(d_in=64, d_out=64, rank=4)` (LoRA step-0 parity, freeze check & parameter savings) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Part VI Capstone Exam Grader
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import math
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import sharukh_tutor as tutor

torch.set_num_threads(1)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """Provided Chapter 32 helper (To-Do 32.1)."""
    d_k = Q.size(-1)
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    return attn @ V, attn


def sinusoidal_positional_encoding(max_len, d_model):
    """Provided Chapter 32 helper (To-Do 32.2)."""
    pe = torch.zeros(max_len, d_model, dtype=torch.float32)
    pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float32)
        * (-math.log(10000.0) / d_model)
    )
    pe[:, 0::2] = torch.sin(pos * div_term)
    pe[:, 1::2] = torch.cos(pos * div_term[: d_model // 2])
    return pe


class MultiHeadSelfAttention(nn.Module):
    """Provided Chapter 32 helper (To-Do 32.3)."""

    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must divide by num_heads")
        self.d_model, self.num_heads = d_model, num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        H, dk = self.num_heads, self.d_k
        Q = self.W_q(x).view(B, T, H, dk).transpose(1, 2)
        K = self.W_k(x).view(B, T, H, dk).transpose(1, 2)
        V = self.W_v(x).view(B, T, H, dk).transpose(1, 2)
        ctx, attn = scaled_dot_product_attention(Q, K, V, mask=mask)
        merged = ctx.transpose(1, 2).contiguous().view(
            B, T, self.d_model
        )
        return self.W_o(merged), attn


class TransformerEncoderBlock(nn.Module):
    """Provided Chapter 32 helper (To-Do 32.4)."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x, mask=None):
        attn_out, weights = self.attn(x, mask=mask)
        x = self.norm1(x + self.drop(attn_out))
        x = self.norm2(x + self.drop(self.ffn(x)))
        return x, weights


class ClinicalVocabTokenizer:
    """Provided Chapter 32 helper (To-Do 32.5)."""

    PAD_ID = 0
    UNK_ID = 1

    def __init__(self):
        self.token_to_id = {"<PAD>": 0, "<UNK>": 1}

    def fit(self, texts):
        words = sorted(
            {w for t in texts for w in t.lower().split()}
        )
        self.token_to_id = {"<PAD>": 0, "<UNK>": 1}
        for idx, word in enumerate(words, start=2):
            self.token_to_id[word] = idx
        return self

    def encode(self, text, max_len):
        ids = [
            self.token_to_id.get(w, self.UNK_ID)
            for w in text.lower().split()
        ][:max_len]
        return ids + [self.PAD_ID] * (max_len - len(ids))

    def __len__(self):
        return len(self.token_to_id)


class LoRALinear(nn.Module):
    """Provided Chapter 32 helper (To-Do 32.6)."""

    def __init__(self, base_linear, rank=4, alpha=8.0):
        super().__init__()
        self.base_linear = base_linear
        for p in self.base_linear.parameters():
            p.requires_grad = False
        self.rank = rank
        self.alpha = float(alpha)
        self.scaling = float(alpha) / rank
        self.lora_A = nn.Parameter(
            torch.randn(rank, base_linear.in_features) * 0.02
        )
        self.lora_B = nn.Parameter(
            torch.zeros(base_linear.out_features, rank)
        )

    def forward(self, x):
        delta = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return self.base_linear(x) + delta


tutor.activate(chapter="ch32", notebook="coding_exam")


---
## Exam Task 32.1 (30 Points): `verify_causal_attention_mask(seq_len)`

Audit autoregressive causal masking:
1. Build a lower-triangular allow-mask `torch.tril(torch.ones(1, seq_len, seq_len))`.
2. Run `scaled_dot_product_attention` on random `(1, seq_len, 8)` `Q, K, V` tensors.
3. Measure `upper_triangle_max` (max weight strictly above the main diagonal) and `row_sum_error` (max `|row_sum - 1.0|`).
4. Return `{"upper_triangle_max": float, "row_sum_error": float, "is_causal": bool}`.

In [ ]:
def verify_causal_attention_mask(seq_len):
    """Audit lower-triangular causal attention masking on seq_len."""
    torch.manual_seed(0)
    Q = torch.randn(1, seq_len, 8)
    K = torch.randn(1, seq_len, 8)
    V = torch.randn(1, seq_len, 8)
    mask = torch.tril(torch.ones(1, seq_len, seq_len))
    _, attn = scaled_dot_product_attention(Q, K, V, mask=mask)
    if seq_len > 1:
        upper_max = float(
            torch.triu(attn[0], diagonal=1).max().item()
        )
    else:
        upper_max = 0.0
    row_err = float((attn.sum(dim=-1) - 1.0).abs().max().item())
    is_causal = bool(upper_max < 1e-6 and row_err < 1e-5)
    return {
        "upper_triangle_max": upper_max,
        "row_sum_error": row_err,
        "is_causal": is_causal,
    }


---
## Exam Task 32.2 (35 Points): `train_clinical_note_transformer(seed=42, epochs=25)`

Build and train an end-to-end clinical triage Transformer classifier offline on CPU:
1. Construct a deterministic **180-note** dataset (`90` Routine=`0` vs. `90` Urgent=`1` clinical symptom notes).
2. Perform a stratified **75/25 split** (`135` train / `45` val) and fit `ClinicalVocabTokenizer` on the training notes (`max_len=10`).
3. Train an `Embedding + SinusoidalPE + TransformerEncoderBlock + MaskedMeanPool + Linear(d_model, 1)` classifier for `epochs` epochs using `BCEWithLogitsLoss` and `Adam`.
4. Return `{"val_acc": round4, "val_auc": round4, "attention_shape": tuple_4d}` achieving **`val_auc >= 0.90`**.

In [ ]:
def train_clinical_note_transformer(seed=42, epochs=25):
    """Train a 180-note clinical triage Transformer on CPU."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    shared = ["patient", "ward", "reports", "clinic", "history"]
    routine = [
        "mild", "stable", "refill", "followup", "normal",
        "routine", "afebrile", "checkup", "rash", "cleared",
    ]
    urgent = [
        "severe", "chest", "pain", "dyspnea", "hypoxia",
        "sepsis", "tachycardia", "syncope", "bleeding", "emergency",
    ]
    texts, labels = [], []
    for _ in range(90):
        ctx = list(rng.choice(shared, size=2, replace=False))
        r_tok = list(rng.choice(routine, size=4, replace=False))
        u_tok = list(rng.choice(urgent, size=4, replace=False))
        if rng.random() < 0.2:
            r_tok[0] = str(rng.choice(urgent))
        if rng.random() < 0.2:
            u_tok[0] = str(rng.choice(routine))
        rn, un = ctx + r_tok, ctx + u_tok
        rng.shuffle(rn)
        rng.shuffle(un)
        texts.extend([" ".join(rn), " ".join(un)])
        labels.extend([0, 1])
    y = np.array(labels, dtype=np.int64)

    tr_t, va_t, y_tr, y_va = train_test_split(
        texts, y, test_size=0.25, stratify=y, random_state=seed
    )
    tok = ClinicalVocabTokenizer().fit(tr_t)
    X_tr = torch.tensor([tok.encode(t, 10) for t in tr_t])
    X_va = torch.tensor([tok.encode(t, 10) for t in va_t])
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)

    class _TriageNet(nn.Module):
        def __init__(self, v_size):
            super().__init__()
            self.emb = nn.Embedding(v_size, 16, padding_idx=0)
            self.register_buffer(
                "pe", sinusoidal_positional_encoding(10, 16)
            )
            self.enc = TransformerEncoderBlock(16, 2, 32, 0.1)
            self.head = nn.Linear(16, 1)

        def forward(self, ids):
            pm = (ids != 0).float()
            am = pm.unsqueeze(1).unsqueeze(2)
            x = self.emb(ids) + self.pe[: ids.size(1)]
            h, attn = self.enc(x, mask=am)
            w = pm.unsqueeze(-1)
            pooled = (h * w).sum(dim=1) / w.sum(dim=1).clamp(min=1.0)
            return self.head(pooled), attn

    model = _TriageNet(len(tok))
    opt = torch.optim.Adam(model.parameters(), lr=0.02)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        logits, _ = model(X_tr)
        loss_fn(logits, y_tr_t).backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        va_logits, va_attn = model(X_va)
        probs = torch.sigmoid(va_logits).squeeze(-1).numpy()
    val_acc = round(float(((probs >= 0.5) == y_va).mean()), 4)
    val_auc = round(float(roc_auc_score(y_va, probs)), 4)
    return {
        "val_acc": val_acc,
        "val_auc": val_auc,
        "attention_shape": tuple(va_attn.shape),
    }


---
## Exam Task 32.3 (35 Points): `audit_lora_parameter_efficiency(d_in=64, d_out=64, rank=4)`

Audit LoRA parameter efficiency, step-0 parity, and gradient freezing:
1. Wrap `base = nn.Linear(d_in, d_out)` in `lora = LoRALinear(base, rank=rank, alpha=2.0 * rank)`.
2. Verify step-0 output parity on a random input `x`: `max_diff < 1e-6`.
3. Run a dummy backward pass verifying `base.weight.grad is None` while `lora.lora_A.grad is not None and lora.lora_B.grad is not None`.
4. Compute `base_params`, `lora_trainable_params`, and `trainable_pct = round(100.0 * lora_trainable_params / (base_params + lora_trainable_params), 2)`.
5. Return `{"base_params": int, "lora_trainable_params": int, "trainable_pct": round2, "initial_parity_ok": bool, "base_frozen_ok": bool}`.

In [ ]:
def audit_lora_parameter_efficiency(d_in=64, d_out=64, rank=4):
    """Audit LoRA parameter count, step-0 parity, and freeze state."""
    torch.manual_seed(0)
    base = nn.Linear(d_in, d_out)
    lora = LoRALinear(base, rank=rank, alpha=2.0 * rank)
    x = torch.randn(4, d_in)
    with torch.no_grad():
        max_diff = (base(x) - lora(x)).abs().max().item()
        parity_ok = bool(max_diff < 1e-6)
    loss = lora(x).pow(2).mean()
    loss.backward()
    frozen_ok = bool(
        base.weight.grad is None
        and lora.lora_A.grad is not None
        and lora.lora_B.grad is not None
    )
    base_params = sum(p.numel() for p in base.parameters())
    lora_params = sum(
        p.numel() for p in lora.parameters() if p.requires_grad
    )
    pct = round(
        100.0 * lora_params / (base_params + lora_params), 2
    )
    return {
        "base_params": int(base_params),
        "lora_trainable_params": int(lora_params),
        "trainable_pct": pct,
        "initial_parity_ok": parity_ok,
        "base_frozen_ok": frozen_ok,
    }


In [ ]:
exam_submissions = {
    "ch32_exam_01": verify_causal_attention_mask,
    "ch32_exam_02": train_clinical_note_transformer,
    "ch32_exam_03": audit_lora_parameter_efficiency,
}

tutor.check_exam("ch32", exam_submissions)


---
## Part VI Capstone Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 32.1 (Strictly Above Diagonal vs. Main Diagonal):** Causal self-attention ALLOWS a token to attend to itself and all earlier positions (`j <= i`), blocking only strictly future tokens (`j > i`). Always extract the upper triangle with `diagonal=1` (`torch.triu(attn[0], diagonal=1)`).
- **Task 32.2 (Masked Mean Pooling Prevents `<PAD>` Dilution):** Averaging all 10 token representations without masking lets 4 trailing `<PAD>` vectors dilute the clinical symptom signal of a 6-word note. Multiplying by `pad_mask.unsqueeze(-1)` and dividing by the true token count stabilizes training and boosts validation AUC above `0.95`.
- **Task 32.3 (Why `trainable_pct` Shrinks Dramatically at Foundation Scale):** For a toy `64x64` layer with `rank=4`, LoRA trains `512 / 4,672 = 10.96%` of total weights (>89% savings). For a real 7B LLM projection (`4096x4096`, `rank=8`), LoRA trains only `65,536 / 16,842,752 = 0.39%`—enabling single-GPU clinical fine-tuning!